# Figure 8.2: Linear to Polynomial Fit
Only the fit view is included here (no contour and no optimization-MSE panels).

### Optional dependency setup
Run this cell only if imports fail in your environment.

In [ ]:
import importlib.util

packages = [
    'numpy', 'matplotlib', 'plotly', 'ipywidgets', 'scipy', 'pandas', 'sklearn', 'seaborn'
]
missing = [p for p in packages if importlib.util.find_spec(p) is None]
if missing:
    print('Installing missing packages:', missing)
    get_ipython().run_line_magic('pip', 'install -q ' + ' '.join(missing))
else:
    print('All common packages already available.')


In [ ]:
import numpy as np
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display

N = 36
x = np.linspace(-2.2, 2.2, N)
noise_base = np.array([0.8*np.sin(7.123*(i+1)) + 0.25*np.cos(3.337*(i+1)) for i in range(N)])

def y_true(v):
    return 0.5*v + 1.2*np.sin(1.1*v) + 0.2*v*v

def fit_poly(xv, yv, degree):
    X = np.vander(xv, N=degree+1, increasing=True)
    A = X.T @ X + 1e-8*np.eye(degree+1)
    b = X.T @ yv
    return np.linalg.solve(A, b)

def eval_poly(theta, xv):
    X = np.vander(xv, N=len(theta), increasing=True)
    return X @ theta

out = widgets.Output()

deg = widgets.IntSlider(description='degree', min=1, max=10, step=1, value=2, continuous_update=False)
noise = widgets.FloatSlider(description='noise', min=0.05, max=1.0, step=0.05, value=0.25, readout_format='.2f', continuous_update=False)

def render(*_):
    y = y_true(x) + noise.value * noise_base
    theta = fit_poly(x, y, deg.value)
    xg = np.linspace(-2.5, 2.5, 240)
    yg = eval_poly(theta, xg)

    fig = go.Figure()
    fig.add_scatter(x=x, y=y, mode='markers', name='data')
    fig.add_scatter(x=xg, y=yg, mode='lines', name='poly fit', line=dict(color='#dc2626', width=3))
    fig.update_layout(
        title=f'Linear to polynomial (degree={deg.value})',
        xaxis=dict(title='x', range=[-2.5, 2.5], autorange=False),
        yaxis=dict(title='y', range=[-4.5, 3.5], autorange=False),
        height=520
    )
    with out:
        out.clear_output(wait=True)
        fig.show()

deg.observe(render, 'value')
noise.observe(render, 'value')
display(widgets.HBox([deg, noise]))
display(out)
render()
